In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10
        

In [10]:
# Datasets and dataloaders
from torch.utils.data import DataLoader         
import torchvision.transforms as transforms
    
#  image => scale(0,1) => normalize(-1.1)
transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

c:\Users\preet\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [11]:
trainloader =  DataLoader(trainset,batch_size=64,shuffle=True)
testloader =  DataLoader(testset,batch_size=64)


# Build the CNN

In [12]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(32,64 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128 ,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
            
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
                
            nn.Linear(256,10)
        )
                
            
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)  #flattening
        x = self.fc_layers(x)
        
        return x
        
            
        

In [ ]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [15]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()
        
        epoch_training_loss += loss.item()
        
    print(f"epoch = {epoch+1}/{epochs} , loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 , loss = 1.3800706015828321
epoch = 2/10 , loss = 0.9517560653064562
epoch = 3/10 , loss = 0.7760121490796814
epoch = 4/10 , loss = 0.6455878994196577
epoch = 5/10 , loss = 0.5380360411332391
epoch = 6/10 , loss = 0.4423569755442917
epoch = 7/10 , loss = 0.3569683192102501
epoch = 8/10 , loss = 0.282432365371748
epoch = 9/10 , loss = 0.22233070951917439
epoch = 10/10 , loss = 0.17166143962089211


In [16]:
#  Evaluate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        
        
print(f"accuracy  =  {correct_labels/total_labels * 100}")

accuracy  =  75.36
